In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('playground-series-s6e7')

print("Path to competition files:", path)

100%|██████████████████████████████████████| 22.3M/22.3M [00:23<00:00, 1.01MB/s]

Extracting files...


Path to competition files: /Users/jaquesgillis/.cache/kagglehub/competitions/playground-series-s6e7


In [4]:
from setup import *
import os

os.listdir(path)

['test.csv', 'train.csv', 'sample_submission.csv']

In [52]:
df = pd.read_csv(path + '/train.csv')

In [73]:
X = df.copy()
X.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


## Expectations

Most of the independent variables here should have a strong relationship with the target (health_condition). One would expect sleep_duration to have a nearly linear relationship with health_condition up to about 8 or 9 hours, and (if there are longer sleep durations in the data set) to fall again. Similar relationships seem likely with heart_rate, and bmi. We might find a more linear relationship between the target and calorie_expenditure, step_count, exercise_duration, water_intake, stress_level, and sleep_quality. 

It also seems likely that many of these features have significant covariance in certain intervals. Some of them may vary with undocumented health conditions. 

## Procedure

The table needs cleaning, and the categorical variables need to be encoded. So first, we'll take care of NaN's and then we'll ordinal encode stress_level, sleep_quality, and the target and one-hot encode diet_type, smoking_alcohol, and gender. Finally, we'll separate the target and train a baseline model before beginning deeper exploration of the features.

In [74]:
# Find columns with null values

print(f"length of table: {len(X)}")

features = X.columns

for feature in features:
    if X[feature].isnull().values.any():
        print(f"{X[feature].isnull().sum()/len(X)} proportion null rows for {feature}")

length of table: 690088
0.1101294327679948 proportion null rows for sleep_duration
0.011350726284184046 proportion null rows for heart_rate
0.020139460474606137 proportion null rows for bmi
0.07658878287986459 proportion null rows for calorie_expenditure
0.020165544104520004 proportion null rows for step_count
0.010000173890866092 proportion null rows for exercise_duration
0.06300210987584193 proportion null rows for water_intake
0.010000173890866092 proportion null rows for diet_type
0.12000063759984234 proportion null rows for stress_level
0.0845269009169845 proportion null rows for sleep_quality
0.05306714505976049 proportion null rows for physical_activity_level
0.04141790612211776 proportion null rows for smoking_alcohol
0.030971412341614404 proportion null rows for gender


As this is synthetic data, the patterns in where the null values fall may not reflect real phenomena. Nevertheless, we might guess that values for sleep_duration, heart_rate, and other clinical features would be most likely to be recorded when they are pathological. These can probably be imputed by a simple sample mean, or by a larger population mean (e.g., 7 hours sleep per night, based on the US population). smoking_alcohol seems most likely to be underreported, with nulls being unreported positives. null genders could represent a third category, "other."

In [75]:
#########################################################################
# For the inputations below, I'm favoring the "outside" view by taking
# US population statistics when they are available and seem likely to fit
# in a college population.
#########################################################################

# Use the US population mean to replace missing sleep duration values
X['sleep_duration'] = X['sleep_duration'].mask(X['sleep_duration'].isnull(), other=7.0)
# Because of high variability of heart rate statistics, use the 
# sample mean for heart_rate
X['heart_rate'] = X['heart_rate'].mask(X['heart_rate'].isnull(), other=X['sleep_duration'].mean())
# https://dqydj.com gives a median bmi of 25.5 for 18-24 year olds
X['bmi'] = X['bmi'].mask(X['bmi'].isnull(), other=25.5)
# The following seem like local statistics, so I'll take the mean of the sample'
# Using median for quantitative variables and mode for categorical
X['calorie_expenditure'] = X['calorie_expenditure'].mask(X['calorie_expenditure'].isnull(), other=X['calorie_expenditure'].median())
X['step_count'] = X['step_count'].mask(X['step_count'].isnull(), other=X['step_count'].median())
X['exercise_duration'] = X['exercise_duration'].mask(X['exercise_duration'].isnull(), other=X['exercise_duration'].median())
X['water_intake'] = X['water_intake'].mask(X['water_intake'].isnull(), other=X['water_intake'].median())
X['diet_type'] = X['diet_type'].mask(X['diet_type'].isnull(), other=X['diet_type'].mode()[0])
X['stress_level'] = X['stress_level'].mask(X['stress_level'].isnull(), other=X['stress_level'].mode()[0])
X['sleep_quality'] = X['sleep_quality'].mask(X['sleep_quality'].isnull(), other=X['sleep_quality'].mode()[0])
X['physical_activity_level'] = X['physical_activity_level'].mask(X['physical_activity_level'].isnull(), X['physical_activity_level'].mode()[0])
# I'll take a pessimistic view and assume that unreported smoking/drinking means 'yes'
X['smoking_alcohol'] = X['smoking_alcohol'].mask(X['smoking_alcohol'].isnull(), other='yes')
# Create a third category, "other", for null gender values
X['gender'] = X['gender'].mask(X['gender'].isnull(), other="other")

In [76]:
type(X['diet_type'])

pandas.core.series.Series

In [77]:
X = X.drop(columns=['id'])

##################
# One-hot encoding
##################

categorical = ['diet_type', 'smoking_alcohol', 'gender']

one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

for category in categorical:
    # Get names for the one-hot-encoded columns
    new_column_names = X[category].unique()
    # encode
    new_column_data = one_hot.fit_transform(X[[category]])
    # wrap in dataframe
    new_columns = pd.DataFrame(new_column_data)
    # rename
    new_columns.columns = new_column_names
    # attach to the main dataframe
    X = pd.concat([X.drop(columns=category, axis=1), new_columns], axis=1)


In [83]:
X['health_condition'].unique()

array(['unhealthy', 'at-risk', 'fit'], dtype=object)

In [86]:
################
# Ordinal encode
################

ordinal = ['health_condition', 'sleep_quality', 'stress_level', 'physical_activity_level']

ordinal_values = {}
ordinal_values['health_condition'] = ['at-risk', 'unhealthy', 'fit']
ordinal_values['sleep_quality'] = ['poor', 'average', 'good']
ordinal_values['stress_level'] = ['low', 'medium', 'high']
ordinal_values['physical_activity_level'] = ['sedentary', 'moderate', 'active']

for category in ordinal:
    encoder = OrdinalEncoder(categories=[ordinal_values[category]])
    X[category + '_encoded'] = encoder.fit_transform(X[[category]])
    X.drop(columns=[category], inplace=True)

## Baseline model

In [88]:
y = X.pop('health_condition_encoded')

In [89]:
#Train a baseline model

rf_model = RandomForestClassifier(n_estimators=100, max_depth=9)
train_X, val_X, train_y, val_y = train_test_split(X, y)
rf_model.fit(train_X, train_y)
val_predictions = rf_model.predict(val_X)
print(f"Baseline balanced accuracy score: {balanced_accuracy_score(val_y, val_predictions)}")

Baseline balanced accuracy score: 0.8382819991754321
